In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_medallion_dbw"

treatments_bronze = spark.table(
    f"{CATALOG}.bronze.treatments"
)

treatments_bronze.printSchema()

root
 |-- treatment_id: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- treatment_date: date (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _layer: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _record_hash: string (nullable = true)
 |-- _is_duplicate: boolean (nullable = true)
 |-- _raw_row_number: long (nullable = true)



In [0]:
treatments_silver = (
    treatments_bronze

    .dropDuplicates(["treatment_id"])

    .withColumn(
        "appointment_id",
        F.trim(F.col("appointment_id"))
    )

    .withColumn(
        "treatment_type",
        F.initcap(F.trim(F.col("treatment_type")))
    )

    .withColumn(
        "description",
        F.initcap(F.trim(F.col("description")))
    )

    .withColumn(
        "cost",
        F.col("cost").cast("double")
    )

    .withColumn(
        "cost_valid_flag",
        F.when(
            F.col("cost") > 0,
            True
        ).otherwise(False)
    )

    .withColumn(
        "_dq_passed",
        (
            F.col("treatment_id").isNotNull()
            &
            F.col("appointment_id").isNotNull()
            &
            F.col("treatment_type").isNotNull()
            &
            F.col("treatment_date").isNotNull()
            &
            F.col("cost_valid_flag")
        )
    )

    .withColumn(
        "_dq_score",
        (
            F.when(F.col("treatment_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("appointment_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("treatment_type").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("treatment_date").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("cost_valid_flag"), 1).otherwise(0)
        ) / F.lit(5.0)
    )

    .withColumn(
        "_dq_failure_reason",
        F.when(
            F.col("_dq_passed") == False,
            F.lit("Treatment validation failed")
        )
    )

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )

    .withColumn(
        "_enrichment_source",
        F.lit("bronze.treatments")
    )
)

In [0]:
appointment_keys = (
    spark.table(
        f"{CATALOG}.silver.appointments"
    )
    .select("appointment_id")
    .dropDuplicates()
)

treatments_silver = (
    treatments_silver

    .join(
        appointment_keys.withColumn(
            "valid_appointment",
            F.lit(True)
        ),
        on="appointment_id",
        how="left"
    )

    .withColumn(
        "valid_appointment",
        F.coalesce(
            F.col("valid_appointment"),
            F.lit(False)
        )
    )

    .withColumn(
        "_dq_passed",
        F.col("_dq_passed")
        &
        F.col("valid_appointment")
    )
)

In [0]:
treatments_silver = (
    treatments_silver

    .withColumn(
        "treatment_year",
        F.year("treatment_date")
    )

    .withColumn(
        "treatment_month",
        F.month("treatment_date")
    )

    .withColumn(
        "cost_band",
        F.when(
            F.col("cost") < 1000,
            "LOW"
        )
        .when(
            F.col("cost") < 3000,
            "MEDIUM"
        )
        .otherwise(
            "HIGH"
        )
    )
)

In [0]:
(
    treatments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.treatments"
    )
)

In [0]:
print(
    "Silver treatments:",
    spark.table(
        f"{CATALOG}.silver.treatments"
    ).count()
)

display(
    spark.table(
        f"{CATALOG}.silver.treatments"
    )
)

Silver treatments: 200


appointment_id,treatment_id,treatment_type,description,cost,treatment_date,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_ingestion_date,_pipeline_version,_source_system,_record_hash,_is_duplicate,_raw_row_number,cost_valid_flag,_dq_passed,_dq_score,_dq_failure_reason,_silver_load_timestamp,_silver_batch_id,_bronze_batch_id,_enrichment_source,valid_appointment,treatment_year,treatment_month,cost_band
A176,T176,Mri,Advanced Protocol,1096.36,2023-04-26,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,1de1b73b698515c842521500822308a48a0a4f79b8f9221b5193a6847db80d68,false,22,true,true,1.0,null,2026-08-10T16:02:32.822Z,2fd91410-2965-46ea-a4d1-09a500d788e6,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,4,MEDIUM
A116,T116,X-ray,Advanced Protocol,1288.86,2023-07-07,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,5564ac1d221e9a65bc377c2c8484b1d107f8ea99d757526e331c0ea523b22029,false,69,true,true,1.0,null,2026-08-10T16:02:32.822Z,f181120f-f748-49b7-b9a1-8d93efc124e0,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,7,MEDIUM
A139,T139,Mri,Basic Screening,4217.3,2023-10-10,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,6abafbad4d31559db5e3f5cab061d4c5c51ad331594df48d79d01fcbc35b766d,false,89,true,true,1.0,null,2026-08-10T16:02:32.822Z,242516e2-029e-4b17-832b-819490cea404,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,10,HIGH
A124,T124,Chemotherapy,Standard Procedure,3492.1,2023-03-16,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,7b7178f37e329ff35ea99124614503801f4ee9254b935d359aa32cdb18f29fb9,false,103,true,true,1.0,null,2026-08-10T16:02:32.822Z,376832e9-0992-41da-8bef-4637dd6123e3,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,3,HIGH
A007,T007,Chemotherapy,Advanced Protocol,534.03,2023-04-09,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,820a8cba0936c07b93eebff20f99f8812c986f12f6836bbcbbbd8fa7d2832a9b,false,109,true,true,1.0,null,2026-08-10T16:02:32.822Z,e80a43ef-9a4b-45e6-94af-f2b9083500e3,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,4,LOW
A167,T167,Chemotherapy,Basic Screening,1871.06,2023-11-15,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,8c5104931e7de9e95672ab9f285f3548cb6353bb53d8d9a401a5a65ddacd1af8,false,116,true,true,1.0,null,2026-08-10T16:02:32.822Z,2d2479ec-c554-4c4f-a1cb-85d8c2581f31,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,11,MEDIUM
A085,T085,Ecg,Advanced Protocol,968.49,2023-02-18,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,8cf9c5187d11e35151663098a4bdbb07b958ba409813ea36f234bf59b77aba5d,false,118,true,true,1.0,null,2026-08-10T16:02:32.822Z,69cd9655-ba8d-48a7-a2ad-f0364bac9c2c,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.treatments,true,2023,2,LOW
A087,T087,Ecg,Advanced Protocol,3102.74,2023-10-19,2026-08-10T15:37:52.628Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,treatments,8fb432876b51f8fdd47187119ce46cb88464e0287bdb35d5e59a036c052afe8b,false,120,true,true,1.0,null,2026-08-10T16:02:32.822Z,e66c71

In [0]:
(
    spark.table(
        f"{CATALOG}.silver.treatments"
    )
    .groupBy("treatment_type")
    .agg(
        F.count("*").alias("treatment_count"),
        F.round(
            F.avg("cost"),
            2
        ).alias("avg_cost"),
        F.round(
            F.sum("cost"),
            2
        ).alias("total_cost")
    )
    .orderBy(
        F.col("total_cost").desc()
    )
    .show()
)

+--------------+---------------+--------+----------+
|treatment_type|treatment_count|avg_cost|total_cost|
+--------------+---------------+--------+----------+
|  Chemotherapy|             49| 2629.71| 128855.68|
|           Mri|             36| 3224.95| 116098.16|
|         X-ray|             41| 2698.87| 110653.67|
| Physiotherapy|             36| 2761.61|   99418.1|
|           Ecg|             38| 2532.22|  96224.24|
+--------------+---------------+--------+----------+

